# Vertex AI Serverless Custom Training — Word2Vec + XGBoost

**Purpose:** Train a supervised model from a CSV containing exactly three columns:

- `ID` — identifier only; not used as a model feature
- `text` — free text used to train Word2Vec
- `target` — supervised classification label

This notebook follows the same execution pattern shown in the HSBC Confluence documentation:

**Jupyter Notebook → package `task.py` → upload package to GCS → `CustomPythonPackageTrainingJob` → check training status → Model Registry/GCS → check metrics → local prediction → Endpoint deployment → endpoint prediction → undeploy/delete endpoint.**

> **Important:** Run this notebook in the approved Vertex AI Workbench / Jupyter environment for your HSBC use case. Replace the configuration values in the first configuration cell with the values supplied during use-case onboarding. Do not paste secrets into the notebook.

## 0. Prerequisites

Before running the notebook, confirm:

1. Your use case is onboarded for Vertex AI.
2. You have the use-case GCS bucket.
3. You have permission to use the use-case service account.
4. Vertex AI training is enabled for the project.
5. Your notebook has access to GCS and Vertex AI.
6. Your input CSV has exactly these logical fields: `ID`, `text`, `target`.

The notebook creates the training package automatically, so you do **not** need to manually create `setup.py`, `setup.cfg`, `task.py`, or `__init__.py`.

In [ ]:
# ============================================================
# 1. CONFIGURATION — EDIT THIS CELL ONLY
# ============================================================

PROJECT_ID = "YOUR_GCP_PROJECT_ID"
REGION = "europe-west2"

# Use the GCS bucket provided for your HSBC use case.
BUCKET = "gs://YOUR_USE_CASE_BUCKET"

# Service account supplied during use-case onboarding.
USE_CASE_SERVICE_ACCOUNT = "YOUR_USE_CASE_SERVICE_ACCOUNT"

# Path to your local CSV in the Vertex AI Workbench environment.
# Example: "/home/jupyter/my_dataset.csv"
LOCAL_DATASET_PATH = "/home/jupyter/dataset.csv"

# GCS location where the dataset will be uploaded.
DATASET_GCS_URI = f"{BUCKET}/word2vec/data/dataset.csv"

# GCS location used by Vertex AI for model artifacts.
MODEL_DIR = f"{BUCKET}/word2vec/model"
ARTIFACT_DIR = f"{MODEL_DIR}/model"

# Temporary GCS location for the packaged training code.
PACKAGE_GCS_URI = f"{BUCKET}/word2vec/custom.tar.gz"

# ------------------------------------------------------------
# KMS configuration
# ------------------------------------------------------------
# Your Confluence example uses CMEK. Put the approved key
# resource names here. If your onboarding explicitly says
# CMEK is not required, set these to None.
TRAINING_KMS_KEY = None
MODEL_KMS_KEY = None

# Example format from the documentation:
# TRAINING_KMS_KEY = "projects/PROJECT/locations/europe-west2/keyRings/VertexAI/cryptoKeys/vtSharedKey"
# MODEL_KMS_KEY = "projects/PROJECT/locations/europe-west2/keyRings/VertexAI/cryptoKeys/vtSharedKey"

# ------------------------------------------------------------
# Container images
# ------------------------------------------------------------
# These follow the pattern used in the supplied Confluence
# documentation. If your HSBC platform team has supplied a
# newer approved image, change these two values.
TRAIN_IMAGE = "europe-docker.pkg.dev/vertex-ai/training/xgboost-cpu.2-1:latest"
DEPLOY_IMAGE = "europe-docker.pkg.dev/vertex-ai/prediction/xgboost-cpu.2-1:latest"

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------
VECTOR_SIZE = 100
WORD2VEC_WINDOW = 5
WORD2VEC_MIN_COUNT = 2
WORD2VEC_WORKERS = 4
WORD2VEC_EPOCHS = 10

XGB_ROUNDS = 100
XGB_MAX_DEPTH = 6
XGB_LEARNING_RATE = 0.05

TEST_SIZE = 0.20
RANDOM_STATE = 42

# One replica, same simple pattern as the documentation.
REPLICA_COUNT = 1
MACHINE_TYPE = "n1-standard-4"

# Endpoint name.
ENDPOINT_DISPLAY_NAME = "word2vec-xgboost-endpoint"

# Safety: endpoint deployment can incur GCE compute charges.
DEPLOY_ENDPOINT = True

print("Configuration loaded.")
print("Project:", PROJECT_ID)
print("Region:", REGION)
print("Bucket:", BUCKET)
print("Dataset:", DATASET_GCS_URI)
print("Model directory:", MODEL_DIR)


## 2. Install/verify notebook-side dependencies

In [ ]:
# Notebook-side packages.
# The training package has its own requirements in setup.py.

%pip install -q google-cloud-aiplatform google-cloud-storage pandas numpy scikit-learn xgboost gensim


In [ ]:
# Imports and authentication check

import os
import json
import shutil
import tarfile
import pathlib
import textwrap
import subprocess
import sys
import time

import numpy as np
import pandas as pd

from google.cloud import storage
from google.cloud import aiplatform

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

# Initialize Vertex AI exactly as in the documented pattern.
aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=BUCKET,
)

print("Vertex AI initialized successfully.")


## 3. Validate the input dataset before submitting a remote job

In [ ]:
# Load only enough to validate the local input.
df_check = pd.read_csv(LOCAL_DATASET_PATH)

required_columns = ["ID", "text", "target"]

missing = [c for c in required_columns if c not in df_check.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if len(df_check) == 0:
    raise ValueError("Dataset is empty.")

if df_check["text"].isna().all():
    raise ValueError("The text column contains no usable text.")

if df_check["target"].isna().any():
    raise ValueError("Target contains missing values. Clean these before training.")

print("Dataset shape:", df_check.shape)
print("Columns:", list(df_check.columns))
print("\nTarget distribution:")
print(df_check["target"].value_counts(dropna=False))

print("\nSample:")
display(df_check.head())


## 4. Upload the dataset to the use-case GCS bucket

In [ ]:
# Upload the local dataset to GCS.
# This mirrors the documentation's model/data flow:
# local notebook -> use-case GCS -> Vertex AI training job.

storage_client = storage.Client(project=PROJECT_ID)

gcs_path = DATASET_GCS_URI.replace("gs://", "", 1)
bucket_name, blob_name = gcs_path.split("/", 1)

bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(blob_name)

blob.upload_from_filename(LOCAL_DATASET_PATH)

print("Uploaded dataset to:")
print(DATASET_GCS_URI)

# Verify it exists.
print("\nGCS verification:")
print(blob.exists())


## 5. Create the custom training package

This is the equivalent of the Confluence steps where `custom/`, `setup.py`, `setup.cfg`, `PKG-INFO`, `trainer/__init__.py`, and `trainer/task.py` are created.

The notebook generates them automatically.

In [ ]:
# Recreate the package folder from scratch.

PACKAGE_ROOT = pathlib.Path("custom")
TRAINER_DIR = PACKAGE_ROOT / "trainer"

if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)

TRAINER_DIR.mkdir(parents=True)

(PACKAGE_ROOT / "setup.py").write_text('from setuptools import setup, find_packages\n\nsetup(\n    name="word2vec-xgboost-training",\n    version="0.1.0",\n    packages=find_packages(),\n    install_requires=[\n        "gensim==4.3.3",\n        "numpy==1.26.4",\n        "pandas==2.2.2",\n        "scikit-learn==1.5.2",\n        "xgboost==2.1.3",\n        "google-cloud-storage>=2.18,<3",\n        "cloudml-hypertune>=0.1,<1",\n    ],\n)\n', encoding="utf-8")
(PACKAGE_ROOT / "setup.cfg").write_text('[metadata]\nname = word2vec-xgboost-training\nversion = 0.1.0\n\n[options]\npackages = find:\n', encoding="utf-8")
(PACKAGE_ROOT / "README.md").write_text('# Word2Vec + XGBoost Vertex AI Training\n\nTraining entry point: trainer.task\n\nInput CSV columns:\nID, text, target\n\nID is metadata only.\ntext is tokenized and used to train Word2Vec.\nDocument vectors are created by averaging word vectors.\ntarget is used to train the XGBoost classifier.\n', encoding="utf-8")
(TRAINER_DIR / "__init__.py").write_text('# Word2Vec + XGBoost Vertex AI training package.\n', encoding="utf-8")

print("Package structure created:")
for path in sorted(PACKAGE_ROOT.rglob("*")):
    print(path)


In [ ]:
task_code = r'''import argparse
import json
import logging
import os
import re
import tempfile

import numpy as np
import pandas as pd
import xgboost as xgb

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from google.cloud import storage
import hypertune


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--model-dir", required=True, type=str)
    parser.add_argument("--dataset-data-url", required=True, type=str)

    parser.add_argument("--word2vec-vector-size", type=int, default=100)
    parser.add_argument("--word2vec-window", type=int, default=5)
    parser.add_argument("--word2vec-min-count", type=int, default=2)
    parser.add_argument("--word2vec-workers", type=int, default=4)
    parser.add_argument("--word2vec-epochs", type=int, default=10)

    parser.add_argument("--xgb-rounds", type=int, default=100)
    parser.add_argument("--xgb-max-depth", type=int, default=6)
    parser.add_argument("--xgb-learning-rate", type=float, default=0.05)

    parser.add_argument("--test-size", type=float, default=0.20)
    parser.add_argument("--random-state", type=int, default=42)

    return parser.parse_args()


def download_from_gcs(gcs_uri, local_path):
    if not gcs_uri.startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got {gcs_uri}")

    path = gcs_uri[5:]
    bucket_name, blob_name = path.split("/", 1)

    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    logger.info("Downloading %s -> %s", gcs_uri, local_path)
    blob.download_to_filename(local_path)


def upload_to_gcs(local_path, gcs_uri):
    if not gcs_uri.startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got {gcs_uri}")

    path = gcs_uri[5:]
    bucket_name, blob_name = path.split("/", 1)

    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    logger.info("Uploading %s -> %s", local_path, gcs_uri)
    blob.upload_from_filename(local_path)


def tokenize_text(text):
    if pd.isna(text):
        return []

    text = str(text).lower()

    # Unicode-aware word/number tokenization.
    return re.findall(r"\b\w+\b", text, flags=re.UNICODE)


def load_dataset(dataset_path):
    df = pd.read_csv(dataset_path)

    required_columns = {"ID", "text", "target"}
    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    df = df[["ID", "text", "target"]].copy()
    df = df.dropna(subset=["target"])
    df["text"] = df["text"].fillna("").astype(str)

    # Target must be integer class labels for this example.
    df["target"] = df["target"].astype(int)

    if df["target"].nunique() < 2:
        raise ValueError("Target must contain at least two classes.")

    return df


def train_word2vec(
    sentences,
    vector_size,
    window,
    min_count,
    workers,
    epochs,
):
    logger.info("Training Word2Vec.")
    logger.info(
        "vector_size=%s window=%s min_count=%s workers=%s epochs=%s",
        vector_size,
        window,
        min_count,
        workers,
        epochs,
    )

    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        sg=1,
        epochs=epochs,
        seed=42,
    )

    logger.info("Word2Vec vocabulary size=%s", len(model.wv))
    return model


def document_to_vector(tokens, word2vec_model):
    vectors = [
        word2vec_model.wv[token]
        for token in tokens
        if token in word2vec_model.wv
    ]

    if not vectors:
        return np.zeros(
            word2vec_model.vector_size,
            dtype=np.float32,
        )

    return np.mean(vectors, axis=0).astype(np.float32)


def create_document_embeddings(tokenized_text, word2vec_model):
    return np.vstack([
        document_to_vector(tokens, word2vec_model)
        for tokens in tokenized_text
    ])


def train_xgboost(X_train, y_train, X_test, y_test, args):
    classes = np.unique(y_train)

    if len(classes) == 2:
        params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "max_depth": args.xgb_max_depth,
            "learning_rate": args.xgb_learning_rate,
            "seed": args.random_state,
            "tree_method": "hist",
        }
    else:
        params = {
            "objective": "multi:softprob",
            "eval_metric": "mlogloss",
            "num_class": len(classes),
            "max_depth": args.xgb_max_depth,
            "learning_rate": args.xgb_learning_rate,
            "seed": args.random_state,
            "tree_method": "hist",
        }

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    evals_result = {}

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=args.xgb_rounds,
        evals=[(dtrain, "train"), (dtest, "test")],
        evals_result=evals_result,
        verbose_eval=True,
    )

    return model, evals_result


def evaluate_model(model, X_test, y_test):
    probabilities = model.predict(xgb.DMatrix(X_test))

    if probabilities.ndim == 1:
        predictions = (probabilities >= 0.5).astype(int)

        try:
            auc = roc_auc_score(y_test, probabilities)
        except ValueError:
            auc = None
    else:
        predictions = np.argmax(probabilities, axis=1)

        try:
            auc = roc_auc_score(
                y_test,
                probabilities,
                multi_class="ovr",
            )
        except ValueError:
            auc = None

    metrics = {
        "accuracy": float(
            accuracy_score(y_test, predictions)
        ),
        "precision_weighted": float(
            precision_score(
                y_test,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
        "recall_weighted": float(
            recall_score(
                y_test,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
        "f1_weighted": float(
            f1_score(
                y_test,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
        "roc_auc": float(auc) if auc is not None else None,
        "confusion_matrix": confusion_matrix(
            y_test,
            predictions,
        ).tolist(),
    }

    return metrics


def save_artifacts(
    model,
    word2vec_model,
    metrics,
    args,
    output_dir,
):
    os.makedirs(output_dir, exist_ok=True)

    model.save_model(
        os.path.join(output_dir, "model.bst")
    )

    word2vec_model.save(
        os.path.join(output_dir, "word2vec.model")
    )

    word2vec_model.wv.save(
        os.path.join(output_dir, "word_vectors.kv")
    )

    config = {
        "word2vec": {
            "vector_size": args.word2vec_vector_size,
            "window": args.word2vec_window,
            "min_count": args.word2vec_min_count,
            "workers": args.word2vec_workers,
            "epochs": args.word2vec_epochs,
        },
        "document_embedding": {
            "method": "mean_word_vectors",
            "dimension": args.word2vec_vector_size,
        },
        "xgboost": {
            "rounds": args.xgb_rounds,
            "max_depth": args.xgb_max_depth,
            "learning_rate": args.xgb_learning_rate,
        },
    }

    with open(
        os.path.join(output_dir, "training_config.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(config, f, indent=2)

    with open(
        os.path.join(output_dir, "metrics.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(metrics, f, indent=2)


def main():
    args = parse_args()

    logger.info("Starting Word2Vec + XGBoost training.")
    logger.info("Arguments: %s", args)

    with tempfile.TemporaryDirectory() as temp_dir:
        dataset_path = os.path.join(
            temp_dir,
            "dataset.csv",
        )

        download_from_gcs(
            args.dataset_data_url,
            dataset_path,
        )

        df = load_dataset(dataset_path)

        tokenized_text = [
            tokenize_text(text)
            for text in df["text"]
        ]

        # ----------------------------------------------------
        # Split BEFORE Word2Vec training.
        # Word2Vec is trained only on training text.
        # ----------------------------------------------------
        indices = np.arange(len(df))

        train_indices, test_indices = train_test_split(
            indices,
            test_size=args.test_size,
            random_state=args.random_state,
            stratify=df["target"],
        )

        train_sentences = [
            tokenized_text[i]
            for i in train_indices
        ]

        word2vec_model = train_word2vec(
            sentences=train_sentences,
            vector_size=args.word2vec_vector_size,
            window=args.word2vec_window,
            min_count=args.word2vec_min_count,
            workers=args.word2vec_workers,
            epochs=args.word2vec_epochs,
        )

        # Convert every document using the SAME trained W2V model.
        embeddings = create_document_embeddings(
            tokenized_text,
            word2vec_model,
        )

        X_train = embeddings[train_indices]
        X_test = embeddings[test_indices]

        y_train = (
            df.iloc[train_indices]["target"]
            .astype(int)
            .to_numpy()
        )

        y_test = (
            df.iloc[test_indices]["target"]
            .astype(int)
            .to_numpy()
        )

        logger.info("X_train shape=%s", X_train.shape)
        logger.info("X_test shape=%s", X_test.shape)

        model, evals_result = train_xgboost(
            X_train,
            y_train,
            X_test,
            y_test,
            args,
        )

        metrics = evaluate_model(
            model,
            X_test,
            y_test,
        )

        metrics["training_history"] = evals_result

        logger.info("Metrics=%s", metrics)

        local_output_dir = os.path.join(
            temp_dir,
            "model",
        )

        save_artifacts(
            model=model,
            word2vec_model=word2vec_model,
            metrics=metrics,
            args=args,
            output_dir=local_output_dir,
        )

        # Vertex AI supplies AIP_MODEL_DIR.
        # Upload artifacts into that directory.
        aip_model_dir = os.environ.get("AIP_MODEL_DIR")

        if not aip_model_dir:
            raise RuntimeError(
                "AIP_MODEL_DIR was not supplied by Vertex AI."
            )

        for filename in os.listdir(local_output_dir):
            local_file = os.path.join(
                local_output_dir,
                filename,
            )

            destination = (
                aip_model_dir.rstrip("/")
                + "/"
                + filename
            )

            upload_to_gcs(
                local_file,
                destination,
            )

        # Same hyperparameter tuning reporting concept
        # used in the supplied documentation.
        hpt = hypertune.HyperTune()

        hpt.report_hyperparameter_tuning_metric(
            hyperparameter_metric_tag="f1",
            metric_value=metrics["f1_weighted"],
        )

    logger.info("TRAINING COMPLETED.")


if __name__ == "__main__":
    main()
'''
(TRAINER_DIR / "task.py").write_text(task_code, encoding="utf-8")
print("Created:", TRAINER_DIR / "task.py")

## 6. Inspect the generated package before uploading

In [ ]:
for path in sorted(PACKAGE_ROOT.rglob("*")):
    if path.is_file():
        print("\n" + "=" * 80)
        print(path)
        print("=" * 80)
        print(path.read_text(encoding="utf-8")[:4000])


## 7. Package the training code and upload it to GCS

In [ ]:
# Same package-assembly concept as the Confluence documentation.

PACKAGE_TAR = pathlib.Path("custom.tar")
PACKAGE_TAR_GZ = pathlib.Path("custom.tar.gz")

for p in [PACKAGE_TAR, PACKAGE_TAR_GZ]:
    if p.exists():
        p.unlink()

with tarfile.open(PACKAGE_TAR, "w") as tar:
    tar.add(PACKAGE_ROOT, arcname="custom")

with tarfile.open(PACKAGE_TAR_GZ, "w:gz") as tar:
    tar.add(PACKAGE_ROOT, arcname="custom")

# Upload compressed package to the use-case bucket.
package_gcs_path = PACKAGE_GCS_URI.replace("gs://", "", 1)
pkg_bucket_name, pkg_blob_name = package_gcs_path.split("/", 1)

pkg_bucket = storage_client.bucket(pkg_bucket_name)
pkg_blob = pkg_bucket.blob(pkg_blob_name)

pkg_blob.upload_from_filename(str(PACKAGE_TAR_GZ))

print("Package uploaded:")
print(PACKAGE_GCS_URI)

print("\nVerification:")
print(pkg_blob.exists())


## 8. Create the Vertex AI CustomPythonPackageTrainingJob

In [ ]:
# This is the key step matching the Confluence architecture:
# Jupyter notebook -> CustomPythonPackageTrainingJob -> remote Vertex AI training.

job_kwargs = dict(
    display_name="word2vec-xgboost-training",
    python_package_gcs_uri=PACKAGE_GCS_URI,
    python_module_name="trainer.task",
    container_uri=TRAIN_IMAGE,
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=BUCKET,
    model_serving_container_image_uri=DEPLOY_IMAGE,
)

# Add CMEK only if supplied by your onboarding.
if TRAINING_KMS_KEY:
    job_kwargs["training_encryption_spec_key_name"] = TRAINING_KMS_KEY

if MODEL_KMS_KEY:
    job_kwargs["model_encryption_spec_key_name"] = MODEL_KMS_KEY

job = aiplatform.CustomPythonPackageTrainingJob(**job_kwargs)

print("Vertex AI training job object created.")
print(job)


## 9. Submit the remote training job

In [ ]:
training_args = [
    "--model-dir", MODEL_DIR,
    "--dataset-data-url", DATASET_GCS_URI,

    "--word2vec-vector-size", str(VECTOR_SIZE),
    "--word2vec-window", str(WORD2VEC_WINDOW),
    "--word2vec-min-count", str(WORD2VEC_MIN_COUNT),
    "--word2vec-workers", str(WORD2VEC_WORKERS),
    "--word2vec-epochs", str(WORD2VEC_EPOCHS),

    "--xgb-rounds", str(XGB_ROUNDS),
    "--xgb-max-depth", str(XGB_MAX_DEPTH),
    "--xgb-learning-rate", str(XGB_LEARNING_RATE),

    "--test-size", str(TEST_SIZE),
    "--random-state", str(RANDOM_STATE),
]

print("Training arguments:")
for arg in training_args:
    print(arg)


In [ ]:
# Submit asynchronously, as in the documentation where the job
# is submitted and then its state is checked separately.

model = job.run(
    args=training_args,
    replica_count=REPLICA_COUNT,
    machine_type=MACHINE_TYPE,
    base_output_dir=MODEL_DIR,
    service_account=USE_CASE_SERVICE_ACCOUNT,
    sync=False,
)

print("Training job submitted.")
print("Display name:", job.display_name)
print("State:", job.state)


## 10. Check training job status

This corresponds to the documentation's **Check training job** step.

The job normally progresses through pending/running and eventually succeeds or fails.

In [ ]:
# Run this cell repeatedly while the job is running.
print("Job state:", job.state)

try:
    print("Resource name:", job.resource_name)
except Exception:
    pass


In [ ]:
# Optional: wait for completion.
# This cell will block until the remote job finishes.

job.wait()

print("Final job state:", job.state)


## 11. Check the trained model in Vertex AI Model Registry

In [ ]:
# List models registered in the current project/region.
models = aiplatform.Model.list()

print("Number of models:", len(models))

for m in models[:20]:
    print(m.resource_name)


In [ ]:
# The training job object exposes the resulting model when the
# training job was configured to upload model artifacts.

try:
    trained_model = job.get_model()
    print("Trained model:")
    print(trained_model)
    print("\nResource name:")
    print(trained_model.resource_name)
except Exception as e:
    print("Could not retrieve the model directly from job.get_model():")
    print(repr(e))
    print("\nUse the Model.list() output above and select the model created by this run.")


## 12. Check the model artifacts in GCS

In [ ]:
# The documentation checks $MODEL_DIR/model.
# In this notebook the artifacts are uploaded directly under MODEL_DIR.

print("Listing model directory:")
!gsutil ls -r "$MODEL_DIR"
print("\nExpected artifact directory:", ARTIFACT_DIR)


## 13. Check `metrics.json`

In [ ]:
metrics_uri = f"{ARTIFACT_DIR}/metrics.json"

print("Metrics URI:", metrics_uri)

!gsutil cat "$metrics_uri"


## 14. Download the trained Word2Vec model and XGBoost model for local prediction

This corresponds to the documentation's **Predict locally** section.

The important difference is that prediction requires both:

1. `word2vec.model`
2. `model.bst`

because raw text must first be converted into the same document vector representation.

In [ ]:
# Download model artifacts locally.

LOCAL_MODEL_DIR = pathlib.Path("downloaded_model")

if LOCAL_MODEL_DIR.exists():
    shutil.rmtree(LOCAL_MODEL_DIR)

LOCAL_MODEL_DIR.mkdir()

!gsutil cp "$ARTIFACT_DIR/model.bst" "$LOCAL_MODEL_DIR/model.bst"
!gsutil cp "$ARTIFACT_DIR/word2vec.model" "$LOCAL_MODEL_DIR/word2vec.model"
!gsutil cp "$ARTIFACT_DIR/training_config.json" "$LOCAL_MODEL_DIR/training_config.json"
!gsutil cp "$ARTIFACT_DIR/metrics.json" "$LOCAL_MODEL_DIR/metrics.json"

print("Downloaded artifacts:")
for p in LOCAL_MODEL_DIR.iterdir():
    print(p)


In [ ]:
from gensim.models import Word2Vec
import xgboost as xgb

local_w2v = Word2Vec.load(
    str(LOCAL_MODEL_DIR / "word2vec.model")
)

local_xgb = xgb.Booster()
local_xgb.load_model(
    str(LOCAL_MODEL_DIR / "model.bst")
)

print("Word2Vec vocabulary:", len(local_w2v.wv))
print("Word2Vec vector size:", local_w2v.vector_size)
print("XGBoost model loaded successfully.")


In [ ]:
# Same preprocessing used inside trainer/task.py.
import re

def tokenize_text(text):
    if pd.isna(text):
        return []
    return re.findall(r"\\b\\w+\\b", str(text).lower(), flags=re.UNICODE)

def document_to_vector(tokens, word2vec_model):
    vectors = [
        word2vec_model.wv[token]
        for token in tokens
        if token in word2vec_model.wv
    ]
    if not vectors:
        return np.zeros(word2vec_model.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0).astype(np.float32)


In [ ]:
def predict_local_texts(texts):
    tokenized = [
        tokenize_text(text)
        for text in texts
    ]

    embeddings = np.vstack([
        document_to_vector(tokens, local_w2v)
        for tokens in tokenized
    ])

    probabilities = local_xgb.predict(
        xgb.DMatrix(embeddings)
    )

    if probabilities.ndim == 1:
        predictions = (
            probabilities >= 0.5
        ).astype(int)

        return pd.DataFrame({
            "text": texts,
            "prediction": predictions,
            "probability": probabilities,
        })

    predictions = np.argmax(
        probabilities,
        axis=1,
    )

    return pd.DataFrame({
        "text": texts,
        "prediction": predictions,
        "probabilities": list(probabilities),
    })


# Replace these examples with your own texts.
example_texts = [
    "customer made a suspicious payment",
    "the payment was successfully completed",
]

display(predict_local_texts(example_texts))


# 15. Prediction using a Vertex AI Endpoint

This follows the deployment pattern in the Confluence document.

### Important for this Word2Vec solution

The standard XGBoost prediction container can serve the trained `model.bst`, but it does **not** know how to run Gensim Word2Vec.

Therefore this executable endpoint example sends the **already-created 100-dimensional document vector** to the endpoint:

`text → local Word2Vec → 100-dimensional vector → Vertex XGBoost endpoint → prediction`

For an endpoint that accepts raw text directly, we will need a custom prediction container later.

In [ ]:
# ============================================================
# 15A. ENDPOINT SWITCH
# ============================================================

# Keep False while validating training.
# Change to True only when you are ready to create GCE-backed
# endpoint resources and incur endpoint compute charges.
DEPLOY_VECTOR_ENDPOINT = False

endpoint = None

if not DEPLOY_VECTOR_ENDPOINT:
    print("Endpoint deployment is OFF.")


In [ ]:
# ============================================================
# 15B. CREATE ENDPOINT
# ============================================================

if DEPLOY_VECTOR_ENDPOINT:

    trained_model = job.get_model()

    endpoint = aiplatform.Endpoint.create(
        display_name=ENDPOINT_DISPLAY_NAME,
        encryption_spec_key_name=MODEL_KMS_KEY,
        project=PROJECT_ID,
        location=REGION,
    )

    print("Endpoint created:")
    print(endpoint.resource_name)

    print("\nDeploying model. This can take several minutes...")

    trained_model.deploy(
        endpoint=endpoint,
        machine_type=MACHINE_TYPE,
        min_replica_count=1,
        max_replica_count=2,
        sync=True,
    )

    print("\nModel deployed.")
    print("Endpoint:", endpoint.resource_name)

else:
    print("Endpoint creation skipped.")


In [ ]:
# ============================================================
# 15C. SEND A PREDICTION TO THE ENDPOINT
# ============================================================

if DEPLOY_VECTOR_ENDPOINT:

    sample_text = "customer made a suspicious payment"

    tokens = tokenize_text(sample_text)

    vector = document_to_vector(
        tokens,
        local_w2v,
    )

    response = endpoint.predict(
        instances=[
            vector.tolist()
        ]
    )

    print("Input text:")
    print(sample_text)

    print("\nDocument vector dimension:")
    print(len(vector))

    print("\nEndpoint response:")
    print(response)

else:
    print("Endpoint prediction skipped.")


## 16. Raw-text endpoint — next production step

Once the training pipeline is working, the production-grade interface should ideally be:

```text
POST /predict
{
    "instances": [
        {"text": "customer made a suspicious payment"}
    ]
}
```

and the serving container should perform:

```text
raw text
   ↓
same tokenisation
   ↓
load word2vec.model
   ↓
document vector
   ↓
load model.bst
   ↓
prediction
```

That requires a custom prediction container. **Do not send raw text to the standard XGBoost prediction container and expect Word2Vec to run automatically.**

## 18. Housekeeping / delete endpoint after inference

In [ ]:
# Once an endpoint has been deployed and you are finished
# with online inference, remove the deployed model and endpoint.
#
# Example:
#
# endpoint.undeploy_all()
# endpoint.delete()
#
# This is important because endpoint instances consume compute resources.
print("Housekeeping instructions loaded. Endpoint deletion is manual/safety-gated.")


# End-to-end flow completed

## What this notebook now does

```text
Local CSV
   |
   | ID / text / target
   v
Upload CSV to GCS
   |
   v
Create custom/
   |
   +-- setup.py
   +-- setup.cfg
   +-- trainer/__init__.py
   +-- trainer/task.py
   |
   v
custom.tar.gz
   |
   v
Upload package to GCS
   |
   v
Vertex AI CustomPythonPackageTrainingJob
   |
   v
Remote training
   |
   +-- train/test split
   +-- tokenize text
   +-- train Word2Vec on training text
   +-- document-level mean embeddings
   +-- train XGBoost
   +-- evaluate
   +-- save model.bst
   +-- save word2vec.model
   +-- save metrics.json
   |
   v
GCS model artifacts / Model Registry
   |
   v
Local prediction
   |
   v
[Optional production endpoint]
raw text -> Word2Vec -> XGBoost -> prediction
   |
   v
Undeploy + delete endpoint
```

### Model artifacts

The training job produces:

- `model.bst` — XGBoost classifier
- `word2vec.model` — complete Word2Vec model
- `word_vectors.kv` — Word2Vec keyed vectors
- `training_config.json` — model configuration
- `metrics.json` — evaluation metrics

### Critical production rule

The **same Word2Vec model and preprocessing logic used during training must be used during inference**. Do not retrain Word2Vec at prediction time.
